# TariffWise — Footwear Classification Prototype

**Musse Abayneh · MPIM-6750-140 · Module 5 Hands-On**

TariffWise is an AI tariff-classification tool for small and mid-size importers. US importers
are legally responsible for assigning the correct Harmonized Tariff Schedule (HTS) code to each
product, and that responsibility remains with the importer even when a broker files the entry.
The product does not compete on the code alone, since automated classification is now common.
It competes on how clearly it reaches a *defensible* code and documents the reasoning.

This notebook is the Module 5 prototype of one screen flow from the Module 4 design. It takes the
footwear niche, runs the AI-style interview that pins down the facts that change the code, suggests
an HTS heading from a small rule set over Chapter 64, generates a numbered rationale tied to the
answers, and produces an exportable audit record.

**Scope and honesty.** The rule set below is a simplified, illustrative subset of Chapter 64. The
codes are real headings and are plausible for the worked examples, but they are not legally
verified. TariffWise suggests and explains. It does not file the entry. A licensed broker or the
importer makes the final decision.

## 1. Setup

Standard library plus pandas for the history table. Built and tested with a Conda environment
(`conda create -n tariffwise python=3.11 pandas`, then `pip install jupyter`).

In [1]:
from dataclasses import dataclass, field
from datetime import date
import pandas as pd

print("TariffWise prototype ready.")

TariffWise prototype ready.


## 2. The classification knowledge

Footwear sits in HTS Chapter 64. The heading turns on two things above all: what the **outer sole**
is made of, and what the **upper** is made of. A short list of the headings the prototype reasons over:

| Heading | Plain-language scope |
|--------|----------------------|
| 6401 | Waterproof footwear, rubber or plastic soles **and** uppers, not assembled by stitching |
| 6402 | Other footwear with rubber or plastic outer soles **and** uppers |
| 6403 | Footwear with rubber, plastic, or leather soles and **leather** uppers |
| 6404 | Footwear with rubber, plastic, or leather soles and **textile** uppers |
| 6405 | Other footwear |

Within 6404, athletic footwear with rubber or plastic soles falls under **6404.11**, and other
textile-upper footwear falls under **6404.19**. A protective **metal toe cap** changes the analysis,
so the prototype treats that answer as a trigger for expert review rather than guessing.

In [2]:
# Each rule is: a test over the captured facts -> (heading, subheading, description).
# Ordered most specific first. This mirrors how a customs reviewer narrows the essential character.

HEADINGS = {
    "6401.92.9000": "Waterproof footwear, rubber/plastic soles and uppers, covering ankle.",
    "6402.99.3165": "Other footwear, rubber/plastic soles and uppers.",
    "6403.99.6075": "Footwear, rubber/plastic/leather soles, leather uppers.",
    "6404.11.9000": "Footwear, rubber/plastic soles, textile uppers — athletic.",
    "6404.19.9000": "Footwear, rubber/plastic soles, textile uppers — other.",
    "6405.90.9060": "Other footwear, not elsewhere specified.",
}

def classify(facts):
    """Suggest a code, then apply a borderline check that can route to expert review."""
    result = _classify_core(facts)
    # A borderline material split keeps the leading code but asks a human to sign off.
    if facts.get("borderline") and result["code"] is not None:
        result["confidence"] = "medium"
        result["status"] = "Expert review"
        result["rationale"].append(
            "The upper material split is borderline, so the case is routed for expert review.")
    return result

def _classify_core(facts):
    """Take captured facts, return a code, description, rationale, confidence, and status."""
    upper   = facts["upper_material"]      # 'textile' | 'leather' | 'rubber_plastic'
    sole    = facts["outsole_material"]    # 'rubber_plastic' | 'leather'
    athletic = facts["athletic"]           # bool
    ankle    = facts["covers_ankle"]       # bool
    toecap   = facts["metal_toe_cap"]      # bool
    waterproof = facts.get("waterproof", False)

    rationale = []
    status = "Suggested"
    confidence = "high"

    # A protective metal toe cap shifts the analysis. Be honest and route to a human.
    if toecap:
        rationale.append("A protective metal toe cap is present, which changes the subheading analysis.")
        rationale.append("This case is routed for expert review rather than auto-suggested.")
        return {"code": None, "description": "Requires expert review",
                "rationale": rationale, "confidence": "low", "status": "Expert review"}

    # Waterproof rubber/plastic one-piece footwear -> 6401.
    if waterproof and upper == "rubber_plastic" and sole == "rubber_plastic":
        rationale.append("The upper and sole are rubber or plastic and the footwear is waterproof.")
        rationale.append("It is not assembled by stitching, which points to heading 6401.")
        if ankle:
            rationale.append("It covers the ankle, consistent with subheading 6401.92.")
        return {"code": "6401.92.9000", "description": HEADINGS["6401.92.9000"],
                "rationale": rationale, "confidence": confidence, "status": status}

    # Textile uppers with rubber/plastic soles -> 6404.
    if upper == "textile" and sole == "rubber_plastic":
        rationale.append("The upper is mostly textile by surface area.")
        rationale.append("The outer sole is rubber or plastic, which points to heading 6404.")
        rationale.append("The shoe does not cover the ankle." if not ankle
                         else "The shoe covers the ankle.")
        if athletic:
            rationale.append("It is athletic footwear, which falls under subheading 6404.11.")
            code = "6404.11.9000"
        else:
            rationale.append("It is not athletic footwear, which falls under subheading 6404.19.")
            code = "6404.19.9000"
        return {"code": code, "description": HEADINGS[code],
                "rationale": rationale, "confidence": confidence, "status": status}

    # Leather uppers -> 6403.
    if upper == "leather" and sole in ("rubber_plastic", "leather"):
        rationale.append("The upper is mostly leather by surface area.")
        rationale.append("The outer sole is rubber, plastic, or leather, which points to heading 6403.")
        return {"code": "6403.99.6075", "description": HEADINGS["6403.99.6075"],
                "rationale": rationale, "confidence": "medium", "status": status}

    # Rubber/plastic uppers, not waterproof -> 6402.
    if upper == "rubber_plastic" and sole == "rubber_plastic":
        rationale.append("The upper and sole are rubber or plastic and the footwear is not waterproof.")
        rationale.append("This points to heading 6402.")
        return {"code": "6402.99.3165", "description": HEADINGS["6402.99.3165"],
                "rationale": rationale, "confidence": "medium", "status": status}

    # Fallback.
    rationale.append("The captured facts do not match a single heading cleanly.")
    rationale.append("The case is routed for expert review.")
    return {"code": "6405.90.9060", "description": HEADINGS["6405.90.9060"],
            "rationale": rationale, "confidence": "low", "status": "Expert review"}

print("Loaded", len(HEADINGS), "candidate headings and the classification logic.")

Loaded 6 candidate headings and the classification logic.


## 3. The AI classification interview

The interview asks **only the questions that change the code**: upper material, sole material,
ankle height, athletic use, and toe construction. This mirrors how a customs reviewer interrogates
a product to find its essential character, which is the exact point where misclassification happens.

The function below replays a fixed interview for the worked example and shows the **working code**
updating as each fact lands, the same live panel shown on Screen 2 of the design.

In [3]:
def run_interview(product_name, qa_pairs, facts):
    """Print the interview as a transcript and show the working heading update live."""
    print("=" * 64)
    print(f"AI CLASSIFICATION INTERVIEW — {product_name}")
    print("=" * 64)
    captured = {}
    for question, answer, fact_key, fact_value in qa_pairs:
        print(f"\n  TariffWise:  {question}")
        print(f"  Importer:    {answer}")
        captured[fact_key] = fact_value
        # Recompute a rough working heading from what we know so far.
        working = working_heading(captured)
        print(f"     -> working heading: {working}")
    print("\n" + "-" * 64)
    print("Interview complete. All code-relevant facts captured.")
    return facts

def working_heading(captured):
    """A light, partial guess used only to animate the live panel during the interview."""
    if captured.get("upper_material") == "textile" and captured.get("outsole_material") == "rubber_plastic":
        return "6404 (textile upper)"
    if captured.get("upper_material") == "textile":
        return "6404 (likely, sole pending)"
    if captured.get("upper_material") == "leather":
        return "6403 (leather upper)"
    if captured.get("upper_material") == "rubber_plastic":
        return "6401/6402 (rubber upper)"
    return "pending"

## 4. Worked example — women's running shoe

This is the same product the Module 4 design walks through. Running the cell needs no typing. It
plays the interview, suggests the code, and prints the rationale.

In [4]:
shoe_facts = {
    "upper_material": "textile",       # textile, about 70 percent of the surface
    "outsole_material": "rubber_plastic",
    "athletic": True,                  # a running shoe
    "covers_ankle": False,             # sits below the ankle
    "metal_toe_cap": False,
    "waterproof": False,
}

shoe_interview = [
    ("What material covers the largest surface area of the upper?",
     "Textile, about 70 percent", "upper_material", "textile"),
    ("What is the outer sole made of?",
     "Rubber", "outsole_material", "rubber_plastic"),
    ("Is this athletic or sports footwear?",
     "Yes, a running shoe", "athletic", True),
    ("Does the shoe cover the ankle?",
     "No, it sits below the ankle", "covers_ankle", False),
    ("Is there a protective metal toe cap?",
     "No", "metal_toe_cap", False),
]

run_interview("Women's running shoe", shoe_interview, shoe_facts)
result = classify(shoe_facts)

print("\n" + "=" * 64)
print("SUGGESTED CLASSIFICATION")
print("=" * 64)
print(f"  HTS code:     {result['code']}")
print(f"  Description:  {result['description']}")
print(f"  Confidence:   {result['confidence']}")
print(f"  Status:       {result['status']}")
print("\n  Why this code:")
for i, line in enumerate(result["rationale"], 1):
    print(f"    {i}. {line}")

AI CLASSIFICATION INTERVIEW — Women's running shoe

  TariffWise:  What material covers the largest surface area of the upper?
  Importer:    Textile, about 70 percent
     -> working heading: 6404 (likely, sole pending)

  TariffWise:  What is the outer sole made of?
  Importer:    Rubber
     -> working heading: 6404 (textile upper)

  TariffWise:  Is this athletic or sports footwear?
  Importer:    Yes, a running shoe
     -> working heading: 6404 (textile upper)

  TariffWise:  Does the shoe cover the ankle?
  Importer:    No, it sits below the ankle
     -> working heading: 6404 (textile upper)

  TariffWise:  Is there a protective metal toe cap?
  Importer:    No
     -> working heading: 6404 (textile upper)

----------------------------------------------------------------
Interview complete. All code-relevant facts captured.

SUGGESTED CLASSIFICATION
  HTS code:     6404.11.9000
  Description:  Footwear, rubber/plastic soles, textile uppers — athletic.
  Confidence:   high
  Sta

## 5. Ask about the result

Screen 3 lets the importer question the result in plain language. A tiny lookup stands in for that
chat so the prototype can surface the reasoning behind a common challenge.

In [5]:
FOLLOW_UPS = {
    "leather": ("Why not classify it as leather footwear?",
                "The upper is mostly textile by surface area, so the leather heading 6403 does not apply."),
    "waterproof": ("Could this be waterproof footwear under 6401?",
                   "Heading 6401 needs rubber or plastic uppers assembled without stitching. This upper is textile, so 6401 does not apply."),
}

def ask_about(topic):
    q, a = FOLLOW_UPS.get(topic, ("", "No canned answer for that question yet."))
    if q:
        print(f"  Importer:    {q}")
    print(f"  TariffWise:  {a}")

ask_about("leather")

  Importer:    Why not classify it as leather footwear?
  TariffWise:  The upper is mostly textile by surface area, so the leather heading 6403 does not apply.


## 6. The audit record

The differentiation lives here. Every classification and its reasoning is stored as a durable record,
because the importer remains responsible for the code even when a broker files the entry. The record
holds the product details, the captured facts, the suggested code, and the rationale, and it is
exportable as evidence if customs audits the entry.

In [6]:
# The importer reviews the suggestion on Screen 3 and clicks "Accept and save to audit trail".
accepted = dict(result)
accepted["status"] = "Saved"

def audit_record(product_name, facts, result, record_date=None):
    record_date = record_date or date.today()
    fact_labels = {
        "upper_material":   ("Upper material",  {"textile":"Textile (majority)","leather":"Leather (majority)","rubber_plastic":"Rubber or plastic"}),
        "outsole_material": ("Outer sole",      {"rubber_plastic":"Rubber or plastic","leather":"Leather"}),
        "athletic":         ("Athletic use",    {True:"Yes",False:"No"}),
        "covers_ankle":     ("Ankle coverage",  {True:"Yes",False:"No"}),
        "metal_toe_cap":    ("Metal toe cap",   {True:"Yes",False:"No"}),
    }
    line = "-" * 64
    print(line)
    print(f"AUDIT RECORD — {product_name}".ljust(50) + str(record_date))
    print(line)
    print("PRODUCT DETAILS")
    print(f"  Name:      {product_name}")
    print(f"  Category:  Footwear")
    print("\nCAPTURED FACTS")
    for key,(label,mapping) in fact_labels.items():
        if key in facts:
            print(f"  {label+':':18}{mapping.get(facts[key], facts[key])}")
    print("\nFINAL CLASSIFICATION")
    print(f"  HTS code:    {result['code']}")
    print(f"  Description: {result['description']}")
    print(f"  Status:      {result['status']}")
    print("\nRATIONALE")
    for i,r in enumerate(result["rationale"],1):
        print(f"  {i}. {r}")
    print("\nSTEPS TAKEN")
    print("  1 Product entered  ->  2 AI interview completed  ->  3 Code suggested  ->  4 Accepted by user")
    print(line)

audit_record("Women's running shoe", shoe_facts, accepted, date(2026,6,14))

----------------------------------------------------------------
AUDIT RECORD — Women's running shoe               2026-06-14
----------------------------------------------------------------
PRODUCT DETAILS
  Name:      Women's running shoe
  Category:  Footwear

CAPTURED FACTS
  Upper material:   Textile (majority)
  Outer sole:       Rubber or plastic
  Athletic use:     Yes
  Ankle coverage:   No
  Metal toe cap:    No

FINAL CLASSIFICATION
  HTS code:    6404.11.9000
  Description: Footwear, rubber/plastic soles, textile uppers — athletic.
  Status:      Saved

RATIONALE
  1. The upper is mostly textile by surface area.
  2. The outer sole is rubber or plastic, which points to heading 6404.
  3. The shoe does not cover the ankle.
  4. It is athletic footwear, which falls under subheading 6404.11.

STEPS TAKEN
  1 Product entered  ->  2 AI interview completed  ->  3 Code suggested  ->  4 Accepted by user
----------------------------------------------------------------


## 7. Classification history

Screen 4 lists past classifications with their codes, dates, and status. Running the four sample
products through `classify()` builds that table directly, so the history reflects the engine rather
than hard-coded values.

In [7]:
samples = [
    ("Women's running shoe", date(2026,6,14),
     dict(upper_material="textile", outsole_material="rubber_plastic", athletic=True,  covers_ankle=False, metal_toe_cap=False, waterproof=False)),
    ("Leather sandal", date(2026,6,12),
     dict(upper_material="leather", outsole_material="rubber_plastic", athletic=False, covers_ankle=False, metal_toe_cap=False, waterproof=False)),
    ("Canvas sneaker", date(2026,6,10),
     dict(upper_material="textile", outsole_material="rubber_plastic", athletic=False, covers_ankle=False, metal_toe_cap=False, waterproof=False, borderline=True)),
    ("Rubber rain boot", date(2026,6,9),
     dict(upper_material="rubber_plastic", outsole_material="rubber_plastic", athletic=False, covers_ankle=True, metal_toe_cap=False, waterproof=True)),
]

rows = []
for name, d, facts in samples:
    r = classify(facts)
    # Accepted suggestions are saved to the trail; routed cases keep the review flag.
    shown_status = "Saved" if r["status"] == "Suggested" else r["status"]
    rows.append({"Product": name, "HTS code": r["code"], "Date": d.strftime("%d %b %Y"), "Status": shown_status})

history = pd.DataFrame(rows)
history

,Product,HTS code,Date,Status
0,Women's running shoe,6404.11.9000,14 Jun 2026,Saved
1,Leather sandal,6403.99.6075,12 Jun 2026,Saved
2,Canvas sneaker,6404.19.9000,10 Jun 2026,Expert review
3,Rubber rain boot,6401.92.9000,09 Jun 2026,Saved


## 8. Optional — classify your own footwear

Everything above runs without typing. This last cell is interactive for anyone who wants to try a
different shoe. It is safe to skip. Run it and answer the prompts, or leave it unrun.

In [8]:
def interactive_classify():
    print("Describe the footwear. Press Enter to accept the default in brackets.\n")
    def ask(q, options, default):
        raw = input(f"{q} {options} [{default}]: ").strip().lower()
        return raw if raw else default
    facts = {
        "upper_material":   ask("Upper material?", "(textile/leather/rubber_plastic)", "textile"),
        "outsole_material": ask("Outer sole material?", "(rubber_plastic/leather)", "rubber_plastic"),
        "athletic":         ask("Athletic footwear?", "(yes/no)", "no") in ("yes","y","true"),
        "covers_ankle":     ask("Covers the ankle?", "(yes/no)", "no") in ("yes","y","true"),
        "metal_toe_cap":    ask("Protective metal toe cap?", "(yes/no)", "no") in ("yes","y","true"),
        "waterproof":       ask("Waterproof?", "(yes/no)", "no") in ("yes","y","true"),
    }
    r = classify(facts)
    print("\nSuggested code:", r["code"], "—", r["description"])
    print("Status:", r["status"], "| Confidence:", r["confidence"])
    print("Rationale:")
    for i,line in enumerate(r["rationale"],1):
        print(f"  {i}. {line}")

# Uncomment the next line to run the interactive interview:
# interactive_classify()

---

**Decision support only.** TariffWise provides classification research. The rule set here is a
simplified, illustrative subset of HTS Chapter 64 for prototype purposes and is not legal advice.
The importer or a licensed broker makes the final classification decision.